# Soft Actor-Critic (SAC) on Humanoid with Stable-Baselines3

In [ ]:
import gymnasium as gym
import numpy as np
import time
from matplotlib import pyplot as plt

from stable_baselines3 import SAC
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

## Configuration

These are the settings of the [RL Baselines3 Zoo](https://github.com/DLR-RM/rl-baselines3-zoo/blob/master/hyperparams/sac.yml) for MuJoCo environments: the default values of SB3's `SAC` plus 10,000 random steps at the start, written out so that they are easy to inspect and change. The RL Zoo trains Humanoid for 2 million steps; the [Tianshou benchmark](https://github.com/thu-ml/tianshou/tree/master/examples/mujoco) reports a return of about 5500 for SAC after 1 million steps on an earlier version of Humanoid, so we use 1 million.

**Training takes a few hours** (see the Training section).

In [ ]:
# Training configuration
ENV_ID = "Humanoid-v5"
TOTAL_TIMESTEPS = 1000000
LEARNING_STARTS = 10000     # random steps that fill the replay buffer before training
BUFFER_SIZE = 1000000      # capacity of the replay buffer
BATCH_SIZE = 256             # transitions per update
GAMMA = 0.99                 # discount factor
LEARNING_RATE = 0.0003       # actor, critics and entropy temperature
TAU = 0.005                  # soft update of the target critics
N_EVAL_EPISODES = 20         # episodes last up to 1000 steps

SEED = 42

## Environment

`Humanoid-v5` (MuJoCo): a three-dimensional humanoid robot has to walk forward without falling. The observation has 348 entries (positions and velocities of the joints, inertia and velocity of each body part, actuator forces and contact forces), and the action is a vector of 17 **continuous** joint torques in $[-0.4, 0.4]$. The reward at every step is

$$r = \underbrace{5}_{\text{healthy}} + \underbrace{1.25\,\dot x}_{\text{forward}} - \underbrace{0.1\,\|a\|^2}_{\text{control}} - \underbrace{5\times10^{-7}\,\|F_\text{contact}\|^2}_{\text{contact}},$$

where $\dot x$ is the forward velocity of the centre of mass. The episode **terminates** when the height of the torso leaves the healthy range $[1.0, 2.0]$, that is, when the robot falls, and it is truncated after 1000 steps. A robot that only stays up for the whole episode collects about $5 \times 1000 = 5000$; walking forward adds to that. With random actions it falls after 20 to 40 steps.

SAC is off-policy and learns from a replay buffer, so one environment is enough. MuJoCo returns the observations as 64-bit floats; `DtypeObservation` converts them to 32-bit floats, the precision used by the networks anyway, which halves the memory of the replay buffer (about 2.8 GB for one million transitions). `Monitor` records the return of every training episode.

In [ ]:
env = Monitor(gym.wrappers.DtypeObservation(gym.make(ENV_ID), np.float32))

print("Observation space:", env.observation_space.shape, env.observation_space.dtype)
print("Action space:", env.action_space)
print("Maximum episode steps:", env.spec.max_episode_steps)
print("Reward threshold:", env.spec.reward_threshold)

## SAC agent

SAC learns a **stochastic** policy that maximizes the return plus a bonus for the entropy of the policy,

$$J(\theta) = \mathbb{E}\Big[\sum_{t} \gamma^t \big(r_{t+1} + \alpha\, \mathcal{H}\big[\pi(\cdot \mid s_t;\theta)\big]\big)\Big],$$

so it keeps exploring by itself and needs no added noise. The temperature $\alpha$ is tuned automatically during training (`ent_coef="auto"`, the default). With `verbose=1`, SB3 prints a table of training statistics during training.

In [ ]:
model = SAC(
    "MlpPolicy",
    env,
    learning_rate=LEARNING_RATE,
    buffer_size=BUFFER_SIZE,
    learning_starts=LEARNING_STARTS,
    batch_size=BATCH_SIZE,
    tau=TAU,
    gamma=GAMMA,
    verbose=1,
    seed=SEED,
    device="cpu",
)

print(model.policy)

The printout shows:

* `actor`: the policy $\pi(a \mid s;\theta)$, two hidden layers of 256 ReLU units followed by two heads, `mu` and `log_std`, the mean and the log standard deviation of a Gaussian over the 17 torques. A sampled action is squashed with $\tanh$ and rescaled to the torque limits.
* `critic`: **two** action-value networks, `qf0` and `qf1`, each receiving the state and the action ($348 + 17 = 365$ inputs). Using the smaller of their two estimates counters the overestimation of Q-learning.
* `critic_target`: slowly moving copies $w_j^-$ of the two critics, updated as $w_j^- \leftarrow \tau w_j + (1-\tau)\, w_j^-$. There is no target actor.

## Training

During the first `LEARNING_STARTS` steps the agent acts at random. After that, at every environment step it stores the transition and performs one update with a minibatch of 256 transitions from the replay buffer:

* **Critics**: both move towards the soft Bellman target

$$y = r + \gamma \Big(\min_{j=0,1} q\big(s', a'; w_j^-\big) - \alpha \ln \pi(a' \mid s';\theta)\Big), \qquad a' \sim \pi(\cdot \mid s';\theta);$$

* **Actor**: maximize $\min_j q(s, a; w_j) - \alpha \ln \pi(a \mid s;\theta)$ with $a \sim \pi(\cdot \mid s;\theta)$, that is, prefer actions with a high value *and* a high entropy;
* **Temperature**: $\alpha$ is adjusted so that the entropy of the policy stays close to a target value, $-17$ (minus the number of torques).

For off-policy algorithms, `log_interval` counts **episodes**; we ask for a table every 500 episodes. The most useful entries are `rollout/ep_rew_mean` and `rollout/ep_len_mean` (mean return and length of the last 100 training episodes), `train/critic_loss`, `train/actor_loss` and `train/ent_coef` (the temperature $\alpha$).

**This cell below can take a few hours**:

In [ ]:
start_time = time.time()
model.learn(total_timesteps=TOTAL_TIMESTEPS, log_interval=500)
training_minutes = (time.time() - start_time) / 60

print(f"Training time: {training_minutes:.2f} minutes")
print(f"Environment steps: {model.num_timesteps}")
print(f"Updates: {model._n_updates}")

## Learning curve

`Monitor` recorded the return and the length of every training episode. The dotted line marks the end of the random steps, and the dashed line the healthy reward of an episode in which the robot stays up for all 1000 steps.

In [ ]:
episode_returns = np.array(env.get_episode_rewards())
episode_end_steps = np.cumsum(env.get_episode_lengths())

window = 20
rolling_returns = np.convolve(episode_returns, np.ones(window) / window, mode="valid")

plt.figure(figsize=(8, 5))
plt.plot(episode_end_steps, episode_returns, alpha=0.3, label="Return per episode")
plt.plot(episode_end_steps[window - 1:], rolling_returns, label=f"Rolling average ({window} episodes)")
plt.axvline(LEARNING_STARTS, color="gray", linestyle=":", label="End of random steps")
plt.axhline(5000, color="tab:red", linestyle="--", label="Healthy reward for 1000 steps (5000)")
plt.xlabel("Environment steps")
plt.ylabel("Return")
plt.title(f"SAC: rolling average return (window = {window}) for sampled (stochastic) policy")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Training episodes: {len(episode_returns)}")

## Evaluate the policy

`evaluate_policy` from SB3 plays complete episodes and returns their returns and lengths. As in the previous notebooks, we evaluate the trained policy over 20 episodes in two ways:

1. **greedy**: `deterministic=True`, the mean of the Gaussian policy;
2. **sampled**: `deterministic=False`, as during training.

Each evaluation uses a new environment seeded with `SEED + 1000`, so both face the same 20 starting states.

In [ ]:
def evaluate(deterministic):
    """Return the returns and lengths of N_EVAL_EPISODES evaluation episodes."""
    eval_env = Monitor(gym.make(ENV_ID))
    eval_env.reset(seed=SEED + 1000)  # later resets continue from this seed
    returns, lengths = evaluate_policy(
        model,
        eval_env,
        n_eval_episodes=N_EVAL_EPISODES,
        deterministic=deterministic,
        return_episode_rewards=True,
    )
    eval_env.close()
    return np.array(returns), np.array(lengths)


greedy_returns, greedy_lengths = evaluate(deterministic=True)
sampled_returns, sampled_lengths = evaluate(deterministic=False)

for name, returns, lengths in [
    ("greedy ", greedy_returns, greedy_lengths),
    ("sampled", sampled_returns, sampled_lengths),
]:
    print(
        f"{name}: mean {returns.mean():7.1f} | std {returns.std():6.1f} | "
        f"min {returns.min():7.1f} | max {returns.max():7.1f} | "
        f"mean length {lengths.mean():6.1f}"
    )

## One greedy rollout

We record one greedy episode and plot the height of the torso and its forward position $x$, read from the MuJoCo state `rollout_env.unwrapped.data.qpos`. A successful policy keeps the torso height inside the healthy range while $x$ increases steadily.

In [ ]:
rollout_env = gym.make(ENV_ID)
state, info = rollout_env.reset(seed=SEED + 2000)
torso_heights = [rollout_env.unwrapped.data.qpos[2]]
torso_x = [rollout_env.unwrapped.data.qpos[0]]
rollout_return = 0.0

for step in range(rollout_env.spec.max_episode_steps):
    action, _ = model.predict(state, deterministic=True)
    state, reward, terminated, truncated, info = rollout_env.step(action)
    torso_heights.append(rollout_env.unwrapped.data.qpos[2])
    torso_x.append(rollout_env.unwrapped.data.qpos[0])
    rollout_return += reward

    if terminated or truncated:
        break

time_steps = np.arange(len(torso_heights))

fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)
axes[0].plot(time_steps, torso_heights)
axes[0].axhline(1.0, color="red", linestyle="--", label="Healthy range")
axes[0].axhline(2.0, color="red", linestyle="--")
axes[0].set_ylabel("Torso height (m)")
axes[0].grid(True)
axes[0].legend()

axes[1].plot(time_steps, torso_x)
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Torso position x (m)")
axes[1].grid(True)

fig.suptitle(f"Greedy Rollout (return = {rollout_return:.0f})")
fig.tight_layout()
plt.show()

print(f"Rollout return: {rollout_return:.0f}")
print(f"Rollout steps:  {len(torso_heights) - 1}")

## Save the trained agent

`model.save` writes a `.zip` file with the actor, the critics, their target copies and the hyperparameters (not the replay buffer, which would take several gigabytes), and `SAC.load` rebuilds the whole agent from it. We check that the reloaded agent chooses the same greedy action.

To watch the saved agent, run from this folder:

```
python run_sac_humanoid_policy.py --episodes 3
```

In [ ]:
MODEL_PATH = "sac_humanoid_policy"
model.save(MODEL_PATH)  # writes sac_humanoid_policy.zip

# Check that the saved agent can be loaded.
check_model = SAC.load(MODEL_PATH, device="cpu")

check_state, info = rollout_env.reset(seed=SEED + 3000)
original_action, _ = model.predict(check_state, deterministic=True)
reloaded_action, _ = check_model.predict(check_state, deterministic=True)

print("Saved and reloaded:", MODEL_PATH + ".zip")
print("Greedy actions match:", bool(np.allclose(original_action, reloaded_action)))
env.close()
rollout_env.close()